# Analisis Cuaca & Meteorologi Spasial Kebumen
### Integrasi Data Reanalisis ERA5-Land (Hourly) & Curah Hujan CHIRPS (Daily/Monthly)

Notebook ini menggabungkan dataset reanalisis atmosfer **ERA5-Land** dan observasi satelit presipitasi **CHIRPS** dari Google Earth Engine (GEE) yang telah diunduh di folder `data/` untuk menghasilkan analisis cuaca dan meteorologi terpadu di Kabupaten Kebumen:

1. **Data Curah Hujan:** CHIRPS (`precipitation` mm/hari)
2. **Data Meteorologi Atmosfer (ERA5-Land):**
   - Suhu Udara 2m (`temperature_2m` °C)
   - Suhu Titik Embun 2m (`dewpoint_temperature_2m` °C)
   - Kelembapan Relatif / RH (`relative_humidity` % - *diderivasi via August-Roche-Magnus*)
   - Komponen Angin Zonal & Meridional (`u_wind_10m`, `v_wind_10m` m/s)
   - Kecepatan Angin (`wind_speed` m/s - *diderivasi via vektor U & V*)
   - Tekanan Permukaan (`surface_pressure` hPa)

### Fitur Grafik & Visualisasi:
- **Laporan Meteorologi Bulanan Hyetograph HD 3-Panel:** Hujan Harian, Akumulasi Hujan, Suhu Max/Avg/Min.
- **Boxplot Suhu Harian Bulanan:** Sebaran variabilitas suhu per-jam untuk setiap hari dalam sebulan.
- **Heatmap Anomali Suhu Harian (Hari x Bulan):** Matriks anomali suhu terhadap baseline klimatologi.
- **Peta Spasial Multi-Variabel Kebumen:** Distribusi spasial resolusi tinggi dengan overlay batas kecamatan (`33.05_kecamatan.geojson`).
- **Penyimpanan Otomatis:** Seluruh grafik tersimpan dalam resolusi cetak tinggi (300 DPI).

In [ ]:
import os
import glob
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = "DejaVu Sans"
plt.rcParams['font.family'] = "sans-serif"

# ==========================================
# 0. PENGATURAN PATH DINAMIS (LOKAL & KAGGLE)
# ==========================================
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if IS_KAGGLE:
    base_kaggle = Path("/kaggle/input/datasets/jerismeteo")
    
    # Path ERA5-Land
    if (base_kaggle / "gee-era5-land-kebumen/data/era5_land").exists():
        dir_era5 = base_kaggle / "gee-era5-land-kebumen/data/era5_land"
    elif (base_kaggle / "gee-era5-land-kebumen").exists():
        dir_era5 = base_kaggle / "gee-era5-land-kebumen"
    else:
        found_era5 = glob.glob("/kaggle/input/**/era5_land*", recursive=True)
        dir_era5 = Path(found_era5[0]) if found_era5 else Path("/kaggle/working/data/era5_land")
        
    # Path CHIRPS
    if (base_kaggle / "gee-chirps-kebumen/data/chirps").exists():
        dir_chirps = base_kaggle / "gee-chirps-kebumen/data/chirps"
    elif (base_kaggle / "gee-chirps-kebumen/data/chirps_sat").exists():
        dir_chirps = base_kaggle / "gee-chirps-kebumen/data/chirps_sat"
    elif (base_kaggle / "gee-chirps-kebumen").exists():
        dir_chirps = base_kaggle / "gee-chirps-kebumen"
    else:
        found_chirps = glob.glob("/kaggle/input/**/chirps*", recursive=True)
        dir_chirps = Path(found_chirps[0]) if found_chirps else Path("/kaggle/working/data/chirps")
        
    # GeoJSON Batas Kecamatan
    kaggle_geo = base_kaggle / "projek-downscale/33.05_kecamatan.geojson"
    geojson_path = str(kaggle_geo) if kaggle_geo.exists() else glob.glob("/kaggle/input/**/33.05_kecamatan.geojson", recursive=True)[0]
    
    out_base = Path("/kaggle/working/analisis_cuaca_spasial")
else:
    # Path Lokal
    dir_era5 = Path("data/era5_land")
    dir_chirps = Path("data/chirps") if Path("data/chirps").exists() else Path("data/chirps_sat")
    geojson_path = "33.05_kecamatan.geojson"
    out_base = Path("analisis_cuaca_spasial")

out_base.mkdir(parents=True, exist_ok=True)
print(f"📌 Environment       : {'Kaggle' if IS_KAGGLE else 'Lokal'}")
print(f"📌 Folder ERA5-Land  : {dir_era5}")
print(f"📌 Folder CHIRPS     : {dir_chirps}")
print(f"📌 File GeoJSON      : {geojson_path}")
print(f"📌 Folder Output     : {out_base}")


## 1. Memuat Dataset NetCDF (ERA5-Land & CHIRPS)
Membuka file NetCDF multi-tahun menggunakan `xarray.open_mfdataset` dan menyelaraskan rentang waktu.

In [ ]:
files_era5 = sorted(list(dir_era5.glob("**/*.nc")))
files_chirps = sorted(list(dir_chirps.glob("**/*.nc")))

print(f"Total file ERA5-Land ditemukan : {len(files_era5)}")
print(f"Total file CHIRPS ditemukan    : {len(files_chirps)}")

if not files_era5 or not files_chirps:
    raise FileNotFoundError("Pastikan file NetCDF ERA5-Land dan CHIRPS tersedia di direktori data.")

# Buka dataset menggunakan combine='nested' & concat_dim='time' untuk mencegah error koordinat spasial x/y
ds_era5 = xr.open_mfdataset(
    files_era5, 
    combine='nested', 
    concat_dim='time', 
    join='override', 
    compat='override', 
    coords='minimal'
)

ds_chirps = xr.open_mfdataset(
    files_chirps, 
    combine='nested', 
    concat_dim='time', 
    join='override', 
    compat='override', 
    coords='minimal'
)

# ==========================================
# DERIVASI PARAMETER METEOROLOGI TAMBAHAN
# ==========================================
# 1. Kelembapan Relatif / RH (%) via Rumus August-Roche-Magnus
t = ds_era5['temperature_2m']
td = ds_era5['dewpoint_temperature_2m']
es = 6.112 * np.exp((17.625 * t) / (243.04 + t))
e = 6.112 * np.exp((17.625 * td) / (243.04 + td))
ds_era5['relative_humidity'] = xr.DataArray(np.clip((e / es) * 100.0, 0, 100), dims=t.dims, coords=t.coords)

# 2. Kecepatan Angin (m/s) dari Komponen U & V
u = ds_era5['u_wind_10m']
v = ds_era5['v_wind_10m']
ds_era5['wind_speed'] = np.sqrt(u**2 + v**2)

print("\n✅ Dataset ERA5-Land & CHIRPS Berhasil Dimuat!")
print("Variabel ERA5-Land  :", list(ds_era5.data_vars.keys()))
print("Variabel CHIRPS     :", list(ds_chirps.data_vars.keys()))


## 2. Memuat Peta Vektor Wilayah (GeoJSON Kecamatan Kebumen)

In [ ]:
gdf_kec = gpd.read_file(geojson_path)
if gdf_kec.crs != "EPSG:4326":
    gdf_kec = gdf_kec.to_crs("EPSG:4326")

print(f"Berhasil memuat GeoJSON: {len(gdf_kec)} Kecamatan di Kebumen")
gdf_kec.head(3)

## 3. Ekstraksi Deret Waktu Rata-rata Wilayah (*Areal Mean Aggregation*)

In [ ]:
# Ekstraksi Areal Mean untuk ERA5-Land (Per-Jam)
spatial_dims_era5 = [d for d in ds_era5['temperature_2m'].dims if d != 'time']
df_era5_hourly = ds_era5.mean(dim=spatial_dims_era5).to_dataframe()

# Ekstraksi Areal Mean untuk CHIRPS (Harian)
spatial_dims_chirps = [d for d in ds_chirps['precipitation'].dims if d != 'time']
df_chirps_daily = ds_chirps['precipitation'].mean(dim=spatial_dims_chirps).to_series().to_frame(name='precipitation_chirps')

# Agregasi Harian Lengkap untuk Analisis Cuaca
df_daily = pd.DataFrame({
    'temperature_2m': df_era5_hourly['temperature_2m'].resample('D').mean(),
    'temp_max': df_era5_hourly['temperature_2m'].resample('D').max(),
    'temp_min': df_era5_hourly['temperature_2m'].resample('D').min(),
    'relative_humidity': df_era5_hourly['relative_humidity'].resample('D').mean(),
    'rh_max': df_era5_hourly['relative_humidity'].resample('D').max(),
    'rh_min': df_era5_hourly['relative_humidity'].resample('D').min(),
    'surface_pressure': df_era5_hourly['surface_pressure'].resample('D').mean(),
    'wind_speed': df_era5_hourly['wind_speed'].resample('D').mean(),
    'wind_speed_max': df_era5_hourly['wind_speed'].resample('D').max(),
})

# Gabungkan dengan Curah Hujan Harian CHIRPS
df_daily = df_daily.join(df_chirps_daily, how='inner')
df_daily.index.name = 'date'

print("Ringkasan Statistik Harian Gabungan (CHIRPS + ERA5-Land):")
print(df_daily.describe())

## 4. Fungsi 1: Laporan Ringkasan Statistik Meteorologi Bulanan (`analisis_detail`)

In [ ]:
def analisis_detail(df, target_bulan):
    """
    Menghasilkan ringkasan statistik meteorologi mendalam untuk bulan target (YYYY-MM).
    """
    df_filtered = df[df.index.strftime('%Y-%m') == target_bulan]
    if df_filtered.empty:
        print(f"⚠️ Data untuk bulan {target_bulan} tidak ditemukan.")
        return
        
    total_hujan = df_filtered['precipitation_chirps'].sum()
    hujan_maks = df_filtered['precipitation_chirps'].max()
    tgl_maks = df_filtered['precipitation_chirps'].idxmax().strftime('%d %B %Y') if total_hujan > 0 else "-"
    hari_hujan = (df_filtered['precipitation_chirps'] >= 1.0).sum()
    
    suhu_avg = df_filtered['temperature_2m'].mean()
    suhu_max = df_filtered['temp_max'].max()
    suhu_min = df_filtered['temp_min'].min()
    
    rh_avg = df_filtered['relative_humidity'].mean()
    rh_max = df_filtered['rh_max'].max()
    rh_min = df_filtered['rh_min'].min()
    
    sp_avg = df_filtered['surface_pressure'].mean()
    ws_avg = df_filtered['wind_speed'].mean()
    ws_max = df_filtered['wind_speed_max'].max()
    
    print(f"\n{'='*60}")
    print(f"📊 LAPORAN STATISTIK METEOROLOGI KEBUMEN - PERIODE: {target_bulan}")
    print(f"{'='*60}")
    print(f"🌧️ CURAH HUJAN (CHIRPS):")
    print(f"   • Total Akumulasi Bulanan : {total_hujan:.1f} mm")
    print(f"   • Curah Hujan Harian Maks : {hujan_maks:.1f} mm ({tgl_maks})")
    print(f"   • Jumlah Hari Hujan (>=1mm): {hari_hujan} hari")
    print(f"🌡️ SUHU UDARA (ERA5-Land):")
    print(f"   • Rata-rata Bulanan       : {suhu_avg:.1f} °C")
    print(f"   • Suhu Maksimum Absolut   : {suhu_max:.1f} °C")
    print(f"   • Suhu Minimum Absolut   : {suhu_min:.1f} °C")
    print(f"💧 KELEMBAPAN RELATIF (RH):")
    print(f"   • Rata-rata Bulanan       : {rh_avg:.1f} %")
    print(f"   • Rentang (Min - Max)     : {rh_min:.1f}% - {rh_max:.1f}%")
    print(f"💨 ANGIN & TEKANAN:")
    print(f"   • Rata-rata Tekanan Udara : {sp_avg:.1f} hPa")
    print(f"   • Kecepatan Angin Rata-rata: {ws_avg:.1f} m/s")
    print(f"   • Kecepatan Angin Maksimum : {ws_max:.1f} m/s")
    print(f"{'='*60}\n")

## 5. Fungsi 2: Laporan Meteorologi Bulanan Hyetograph HD 3-Panel (`plot_hyetograph_bulanan`)
Menghasilkan visualisasi 3-panel berkualitas cetak tinggi (300 DPI):
1. **Panel Atas:** Diagram batang curah hujan harian (CHIRPS) dengan label nilai.
2. **Panel Tengah:** Garis akumulasi curah hujan kumulatif bulanan.
3. **Panel Bawah:** Profil suhu udara harian (Maksimum, Rata-rata, Minimum) dengan area shading.

In [ ]:
def plot_hyetograph_bulanan(df_daily, target_bulan, output_dir=None):
    """
    Membuat Laporan Meteorologi Hyetograph Bulanan 3-Panel HD.
    """
    df_m = df_daily[df_daily.index.strftime('%Y-%m') == target_bulan].copy()
    if df_m.empty:
        print(f"⚠️ Data tidak tersedia untuk {target_bulan}")
        return
        
    hujan_harian = df_m['precipitation_chirps']
    hujan_kumulatif = hujan_harian.cumsum()
    suhu_avg = df_m['temperature_2m']
    suhu_max = df_m['temp_max']
    suhu_min = df_m['temp_min']
    
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
    x_pos = np.arange(len(df_m))
    tgl_labels = df_m.index.strftime('%d')
    
    # --- PANEL 1: Curah Hujan Harian ---
    bars = ax1.bar(x_pos, hujan_harian, color='royalblue', width=0.65, alpha=0.85, label='Curah Hujan Harian (CHIRPS)')
    for bar, val in zip(bars, hujan_harian):
        if val >= 1.0:
            ax1.annotate(f'{val:.1f}', (bar.get_x() + bar.get_width()/2, val), textcoords='offset points', 
                         xytext=(0, 4), ha='center', fontsize=8, fontweight='bold', color='navy')
    ax1.set_ylabel('Curah Hujan (mm/hari)', fontsize=11, fontweight='bold')
    ax1.legend(loc='upper right', frameon=True)
    ax1.grid(True, linestyle=':', alpha=0.6)
    ax1.set_ylim(0, max(hujan_harian.max() * 1.2, 10))
    
    # --- PANEL 2: Akumulasi Hujan Kumulatif ---
    ax2.plot(x_pos, hujan_kumulatif, color='darkcyan', marker='o', lw=2, markersize=4, label='Akumulasi Curah Hujan')
    ax2.fill_between(x_pos, 0, hujan_kumulatif, color='cyan', alpha=0.15)
    for x, y in zip(x_pos, hujan_kumulatif):
        if x % 5 == 0 or x == len(x_pos) - 1:
            ax2.annotate(f'{y:.1f}', (x, y), textcoords='offset points', xytext=(0, 5), ha='center', fontsize=8, color='darkcyan')
    ax2.set_ylabel('Akumulasi (mm)', fontsize=11, fontweight='bold')
    ax2.legend(loc='upper left', frameon=True)
    ax2.grid(True, linestyle=':', alpha=0.6)
    
    # --- PANEL 3: Suhu Udara (Max, Avg, Min) ---
    ax3.plot(x_pos, suhu_max, color='crimson', marker='^', lw=1.5, markersize=4, label='Suhu Maksimum (°C)')
    ax3.plot(x_pos, suhu_avg, color='limegreen', marker='s', lw=1.5, markersize=4, label='Suhu Rata-rata (°C)')
    ax3.plot(x_pos, suhu_min, color='dodgerblue', marker='v', lw=1.5, markersize=4, label='Suhu Minimum (°C)')
    ax3.fill_between(x_pos, suhu_min, suhu_max, color='orange', alpha=0.15)
    ax3.set_ylabel('Suhu Udara (°C)', fontsize=11, fontweight='bold')
    ax3.legend(loc='lower center', bbox_to_anchor=(0.5, -0.35), ncol=3, frameon=True, fontsize=10)
    ax3.grid(True, linestyle=':', alpha=0.6)
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(tgl_labels, fontsize=10)
    ax3.set_xlabel('Tanggal', fontsize=11, fontweight='bold', labelpad=8)
    
    # Format Judul
    first_dt = df_m.index[0]
    fig.suptitle(f'Laporan Meteorologi Bulanan Kebumen (CHIRPS & ERA5-Land)\nPeriode: {first_dt.strftime("%B %Y")}', 
                 fontsize=15, fontweight='bold', y=0.98)
    plt.tight_layout(pad=1.5)
    fig.subplots_adjust(top=0.91, hspace=0.15)
    
    if output_dir:
        folder = Path(output_dir) / "plots_hyetograph_bulanan" / str(first_dt.year)
        folder.mkdir(parents=True, exist_ok=True)
        save_path = folder / f"Hyetograph_{target_bulan}.png"
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"  ✓ Hyetograph tersimpan di: {save_path}")
        plt.close(fig)
    else:
        plt.show()

## 6. Fungsi 3: Boxplot Variabilitas Suhu Jam-jaman per Hari (`plot_boxplot_suhu`)

In [ ]:
def plot_boxplot_suhu(df_hourly, target_bulan, output_dir=None):
    """
    Membuat Boxplot Variabilitas Suhu Harian (24 jam) dalam 1 bulan.
    """
    df_m = df_hourly[df_hourly.index.strftime('%Y-%m') == target_bulan].copy()
    if df_m.empty:
        print(f"⚠️ Data tidak tersedia untuk {target_bulan}")
        return
        
    df_m['day'] = df_m.index.day
    
    plt.figure(figsize=(15, 6))
    sns.boxplot(data=df_m, x='day', y='temperature_2m', palette='coolwarm')
    
    plt.title(f"Boxplot Variabilitas Suhu Harian ERA5-Land - Kebumen\nPeriode: {target_bulan}", fontsize=14, fontweight='bold')
    plt.xlabel("Tanggal (Hari)", fontsize=11, fontweight='bold')
    plt.ylabel("Suhu Udara (°C)", fontsize=11, fontweight='bold')
    plt.grid(True, linestyle=':', alpha=0.6)
    
    plt.tight_layout()
    
    if output_dir:
        yr = target_bulan.split('-')[0]
        folder = Path(output_dir) / "plots_boxplot_suhu" / str(yr)
        folder.mkdir(parents=True, exist_ok=True)
        save_path = folder / f"Boxplot_Suhu_{target_bulan}.png"
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"  ✓ Boxplot Suhu tersimpan di: {save_path}")
        plt.close()
    else:
        plt.show()

## 7. Fungsi 4: Heatmap Matriks Anomali Suhu Harian (Hari x Bulan) (`plot_heatmap_anomali_suhu`)
Menghitung selisih suhu harian terhadap baseline klimatologi tahunan (WMO baseline) dan memetakannya ke dalam matriks kalender.

In [ ]:
def plot_heatmap_anomali_suhu(df_daily, output_dir=None, cmap="RdBu_r"):
    """
    Membuat Heatmap Anomali Suhu Harian (31 hari x 12 bulan) per tahun terhadap baseline klimatologi.
    """
    df = df_daily.copy()
    df['doy'] = df.index.dayofyear
    df['year'] = df.index.year
    df['month'] = df.index.month
    df['day'] = df.index.day
    
    # Hitung Klimatologi per Hari dalam Setahun (DOY)
    climatology = df.groupby('doy')['temperature_2m'].mean()
    df['temp_anom'] = df['temperature_2m'] - df['doy'].map(climatology)
    
    years = sorted(df['year'].unique())
    nama_bulan = ["Jan","Feb","Mar","Apr","Mei","Jun","Jul","Agu","Sep","Okt","Nov","Des"]
    
    for year in years:
        df_yr = df[df['year'] == year]
        pivot = df_yr.pivot_table(index='day', columns='month', values='temp_anom')
        pivot = pivot.reindex(index=range(1, 32), columns=range(1, 13))
        
        plt.figure(figsize=(14, 12))
        sns.heatmap(pivot, cmap=cmap, center=0, annot=True, fmt=".1f", linewidths=0.5, 
                    linecolor="gray", square=True, cbar_kws={"label": "Anomali Suhu (°C)"})
        
        plt.title(f"Matriks Anomali Suhu Harian Kebumen - Tahun {year}", fontsize=14, fontweight='bold')
        plt.xlabel("Bulan", fontsize=11, fontweight='bold')
        plt.ylabel("Hari / Tanggal", fontsize=11, fontweight='bold')
        plt.xticks(np.arange(12) + 0.5, nama_bulan, rotation=0)
        plt.yticks(rotation=0)
        
        plt.tight_layout()
        
        if output_dir:
            folder = Path(output_dir) / "plots_heatmap_anomali_suhu"
            folder.mkdir(parents=True, exist_ok=True)
            save_path = folder / f"Anomali_Suhu_{year}.png"
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"  ✓ Heatmap Anomali Suhu {year} tersimpan di: {save_path}")
            plt.close()
        else:
            plt.show()

## 8. Fungsi 5: Peta Spasial Multi-Variabel Cuaca Kebumen (`plot_spasial_cuaca_bulanan`)
Membuat peta sebaran spasial 4 panel multi-variabel (Curah Hujan CHIRPS, Suhu Udara, Kelembapan Relatif, Kecepatan Angin) dengan batas administratif kecamatan Kebumen.

In [ ]:
def plot_spasial_cuaca_bulanan(ds_era5, ds_chirps, gdf_kec, target_bulan, output_dir=None):
    """
    Membuat Peta Distribusi Spasial 4-Panel Multi-Variabel Cuaca Kebumen.
    """
    ds_e_m = ds_era5.sel(time=target_bulan)
    ds_c_m = ds_chirps.sel(time=target_bulan)
    
    # Agregasi Spasial Bulanan
    hujan_spasial = ds_c_m['precipitation'].sum(dim='time')
    suhu_spasial = ds_e_m['temperature_2m'].mean(dim='time')
    rh_spasial = ds_e_m['relative_humidity'].mean(dim='time')
    ws_spasial = ds_e_m['wind_speed'].mean(dim='time')
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 16))
    
    # 1. Peta Curah Hujan (CHIRPS)
    hujan_spasial.plot(ax=axes[0, 0], cmap='YlGnBu', cbar_kwargs={'label': 'Curah Hujan Bulanan (mm)'})
    gdf_kec.boundary.plot(ax=axes[0, 0], color='red', linewidth=1)
    axes[0, 0].set_title("Curah Hujan Akumulasi Bulanan (CHIRPS)", fontsize=12, fontweight='bold')
    
    # 2. Peta Suhu Udara Rata-rata
    suhu_spasial.plot(ax=axes[0, 1], cmap='coolwarm', cbar_kwargs={'label': 'Suhu Rata-rata (°C)'})
    gdf_kec.boundary.plot(ax=axes[0, 1], color='black', linewidth=1)
    axes[0, 1].set_title("Suhu Udara Rata-rata (ERA5-Land)", fontsize=12, fontweight='bold')
    
    # 3. Peta Kelembapan Relatif (RH)
    rh_spasial.plot(ax=axes[1, 0], cmap='Blues', cbar_kwargs={'label': 'Kelembapan Relatif (%)'})
    gdf_kec.boundary.plot(ax=axes[1, 0], color='black', linewidth=1)
    axes[1, 0].set_title("Kelembapan Relatif Rata-rata (RH)", fontsize=12, fontweight='bold')
    
    # 4. Peta Kecepatan Angin
    ws_spasial.plot(ax=axes[1, 1], cmap='viridis', cbar_kwargs={'label': 'Kecepatan Angin (m/s)'})
    gdf_kec.boundary.plot(ax=axes[1, 1], color='black', linewidth=1)
    axes[1, 1].set_title("Kecepatan Angin Rata-rata 10m", fontsize=12, fontweight='bold')
    
    for ax in axes.flat:
        ax.set_xlabel("Bujur (Longitude)", fontsize=10)
        ax.set_ylabel("Lintang (Latitude)", fontsize=10)
        
    fig.suptitle(f"Peta Spasial Meteorologi & Cuaca Kabupaten Kebumen\nPeriode: {target_bulan}", fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout(pad=2.0)
    fig.subplots_adjust(top=0.92)
    
    if output_dir:
        yr = target_bulan.split('-')[0]
        folder = Path(output_dir) / "plots_spasial_cuaca" / str(yr)
        folder.mkdir(parents=True, exist_ok=True)
        save_path = folder / f"Peta_Spasial_Cuaca_{target_bulan}.png"
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"  ✓ Peta Spasial Cuaca tersimpan di: {save_path}")
        plt.close(fig)
    else:
        plt.show()

## 9. Eksekusi Batch Pembuatan Plot & Laporan Meteorologi
Menjalankan analisis dan menyimpan seluruh plot ke subfolder terstruktur di `analisis_cuaca_spasial/`.

In [ ]:
# Tentukan Rentang Periode yang akan dianalisis
# Di Lokal: 2020 s.d. selesai, Di Kaggle: Seluruh data yang tersedia
all_dates = pd.to_datetime(df_daily.index)
available_years = sorted(all_dates.year.unique())

start_year = 2020 if (not IS_KAGGLE and 2020 in available_years) else available_years[0]
target_years = [y for y in available_years if y >= start_year]

print(f"📌 Memulai Eksekusi Analisis untuk Tahun: {target_years}")

# 1. Eksekusi Heatmap Anomali Suhu untuk setiap tahun
print("\n--- 1. Menghasilkan Heatmap Anomali Suhu ---")
plot_heatmap_anomali_suhu(df_daily, output_dir=out_base)

# 2. Eksekusi Hyetograph Bulanan, Boxplot Suhu, dan Peta Spasial Cuaca
print("\n--- 2. Menghasilkan Laporan Bulanan (Hyetograph, Boxplot & Peta Spasial) ---")
for yr in target_years:
    print(f"\n>> Memproses Tahun {yr}...")
    months_in_yr = sorted(all_dates[all_dates.year == yr].month.unique())
    
    for m in months_in_yr:
        target_str = f"{yr}-{m:02d}"
        # 1. Tampilkan analisis statistik
        analisis_detail(df_daily, target_str)
        # 2. Hyetograph Bulanan
        plot_hyetograph_bulanan(df_daily, target_str, output_dir=out_base)
        # 3. Boxplot Suhu Harian
        plot_boxplot_suhu(df_era5_hourly, target_str, output_dir=out_base)
        # 4. Peta Spasial Cuaca 4-Panel
        plot_spasial_cuaca_bulanan(ds_era5, ds_chirps, gdf_kec, target_str, output_dir=out_base)

print(f"\n🎉 Seluruh analisis dan grafik cuaca spasial selesai tersimpan di: {out_base.absolute()}")